### PCA and Clustering

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from boardgames_recsys.data.filtering import filter_df
from boardgames_recsys.data.matrix import *
from boardgames_recsys.models.collaborative_filtering import *
from boardgames_recsys.evaluation.ratings import *
from copy import *

%load_ext autoreload
%autoreload 2

In [ ]:
# import DB et set min_reviews

folder = "../database_cleaned"
avis_clean  = pd.read_csv(f"{folder}/avis_clean.csv", index_col=0)
jeux_clean  = pd.read_csv(f"{folder}/jeux_clean.csv", index_col=0)
users       = pd.read_csv(f"{folder}/users.csv", index_col=0)

min_reviews = 20 # change to set one

In [ ]:
# filter data with the minimum reviews
filtered_avis = filter_df(avis_clean, min_reviews)

# create user-game matrix
user_game_ratings, mask_ratings, users_table_assoc, games_table_assoc = get_matrix_user_game(filtered_avis)

In [ ]:
# not centered version 
user_game_ratings.shape

In [ ]:
# first choose the most popular game categories NO CAPITALIZE
categories = jeux_clean["Type"].str.split('|').dropna()
lst_categories = np.unique([item.strip() for list_items in categories for item in list_items])

In [ ]:
# for each categories the number of games
count_cat = np.array([[category, int(jeux_clean['Type'].str.contains(category, na=False, regex=False).sum())] for category in lst_categories])
# two best categories
sorted_categ = count_cat[count_cat[:,1].astype(int).argsort()[::-1]]
sorted_categ

In [ ]:
# for each cat, the game ids for sorted df
jeux_nonan = jeux_clean.dropna()
jeux_nonan = jeux_nonan[jeux_nonan["Game id"].isin(filtered_avis["Game id"].unique())] # games concerned from filtered df

# for each categories, the list of games associated to the tag
cat_games = [[cat, list(jeux_nonan.loc[jeux_nonan['Type'].str.lower().str.contains(cat.lower(), na=False, regex=False)]["Game id"])] for cat in lst_categories]
array_cat_games = np.array(cat_games, dtype=object)[:,1] # list of games id 

In [ ]:
# associated index game
game_ids = pd.Series(np.unique(np.concatenate(array_cat_games)).astype(int)) # game ids concerned for matrix
# associated index users 
users_id = pd.Series(np.unique(filtered_avis["User id"]))
game_ids.shape, users_id.shape


In [ ]:
# using pivot table -> separate cat1|cat2...|catn into many rows, after counting the categories

# merge the databases jeux and avis
avis_jeux = jeux_nonan[["Game id", "Type"]].merge(filtered_avis[["User id", "Game id", "Rating"]], on='Game id')

# for each category create a new row
types = avis_jeux['Type'].str.split('|').explode()
user_game_type = pd.DataFrame({
    'User id': avis_jeux['User id'].repeat(avis_jeux['Type'].str.split('|').apply(len)),
    'Game id': avis_jeux['Game id'].repeat(avis_jeux['Type'].str.split('|').apply(len)),
    'Type': types
})


3 matrix:
- user type count : users and categories, x the number of games of this category user rated, normalised
- user type: users and categories, 1 if rated a game of this category else 0
- game type: games and categories, 1 if the game belongs to this categories else 0, normalized by nb time tags appears

Clustering:
- Hierarchichal clustering
- Bi/Co clustering, clustering based on rows and columns, observe silhouette score

---
User type count matrix

In [ ]:
# create pivot table: rows users, columns categories, x number of reviewed game of user of this categ, else 0, normalised
user_type_count = user_game_type[["User id", "Type"]].pivot_table(index ="User id", columns="Type", aggfunc='size', fill_value=0)
user_type_count = user_type_count.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=1)
user_type_count = user_type_count.div(user_type_count.sum(axis=1), axis=0)

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.metrics import silhouette_score

# dendogram clustering
# figures
plt.figure(figsize=(12,10))
Y = linkage(user_type_count.T, method='ward') # clustering the categories 
_ = dendrogram(Y, orientation='right', labels=user_type_count.columns) 
plt.title("Hierarchical clustering of categories with user_type_count")
plt.ylabel('Game categories')
plt.xlabel('Distance between categories based on users')

# highlight specific labels
highlight_labels = ['Jeux de plateau', 'Jeux de cartes']

for label in plt.gca().get_ymajorticklabels():
    if label.get_text() in highlight_labels:
        label.set_color('red')
        label.set_fontweight('bold')

plt.show()

In [ ]:
# Hierarchical clustering : cluster categories with user-categories-count
from scipy.cluster.hierarchy import fcluster

linkage_matrix = linkage(user_type_count.T, method='ward', optimal_ordering=True)
fl = fcluster(linkage_matrix,3,criterion='maxclust')
plt.title("repartition of categories in cluster")
plt.hist(fl, alpha = 0.6)

for i in range(1, 4):
    print("cluster ", i)
    print(user_type_count.T.index[np.where(fl == i)[0]])

# we get the popular categories amongs users

In [ ]:
# silhouette score evaluation user_type_count.T
scores = [silhouette_score(user_type_count.T,fcluster(linkage(user_type_count.T, method='ward'),i,criterion='maxclust')) for i in range(2,10)]
scores

In [ ]:
# Bi/Coclustering visualisation
from sklearn.cluster import SpectralBiclustering, SpectralCoclustering

fig, ax = plt.subplots(1,2, figsize=(15, 10))

# Co clustering
clustering = SpectralCoclustering(n_clusters=2, random_state=0).fit(user_type_count)
# group the clusters
reordered_rows = user_type_count.iloc[np.argsort(clustering.row_labels_)]
reordered_data = reordered_rows.iloc[:, np.argsort(clustering.column_labels_)]

cax = ax[0].matshow(reordered_data)
ax[0].set_title("Coclustering k=2 user_category_count matrix")
fig.colorbar(cax, ax=ax[0])

# Bi clustering
clustering = SpectralBiclustering(n_clusters=2, random_state=0).fit(user_type_count)
# group the clusters
reordered_rows = user_type_count.iloc[np.argsort(clustering.row_labels_)]
reordered_data = reordered_rows.iloc[:, np.argsort(clustering.column_labels_)]

cax = ax[1].matshow(reordered_data)
ax[1].set_title("Biclustering k=2 user_category_count matrix")
fig.colorbar(cax, ax=ax[1])

plt.subplots_adjust() 
plt.show()

In [ ]:
# silhouette score evaluation user_type_count, co clustering
scores_users_tags = [
    (silhouette_score(user_type_count, model.row_labels_), silhouette_score(user_type_count.T, model.column_labels_)) for i in range(2, 10) for model in [SpectralCoclustering(n_clusters=i, random_state=0).fit(user_type_count)]]
scores_users_tags


In [ ]:
# silhouette score evaluation user_type_count, bi clustering
scores_users_tags = [
    (silhouette_score(user_type_count, model.row_labels_), silhouette_score(user_type_count.T, model.column_labels_)) for i in range(2, 10) for model in [SpectralBiclustering(n_clusters=i, random_state=0).fit(user_type_count)]]
scores_users_tags

In [ ]:

# clustering = SpectralCoclustering(n_clusters=3, random_state=0).fit(user_type_count)
# user_type_count.index[clustering.row_labels_ == 0], user_type_count.index[clustering.row_labels_ == 1], user_type_count.index[clustering.row_labels_ == 2]

After Biclustering the users, we look at the game tags these users rated -> obtained same prop for each cluster

In [ ]:
# bi/coclustering visualise tags frequency among game clusters 
# Nclus = 7
# clustering = SpectralCoclustering(n_clusters=Nclus, random_state=0).fit(user_type_count) 

# fig, axes = plt.subplots(3, 3, figsize=(13, 8))
# clust = 0
# for i in range(3):
#     for j in range(3):
#         if clust >= Nclus:
#             break
        
#         ax = axes[i, j]  # Get the axis for each subplot
        
#         # Filter the DataFrame (clust)
#         cluster_cat = user_type_count.loc[:, (user_type_count.loc[user_type_count.index[np.where(clustering.row_labels_ == clust)]]!= 0).any(axis=0)]
#         print(cluster_cat.shape)
#         # Calculate the frequency of tags in the current cluster
#         pop_tags = ["Jeux de plateau", "Jeux de cartes", "Hasard (Dé, Cartes, ...)", "Placement", "Affrontement", "Combinaison"]
#         cluster_cat = cluster_cat.drop(columns=pop_tags, errors='ignore')
#         tag_frequency = (cluster_cat != 0).sum(axis=0) / (cluster_cat != 0).sum().sum()

#         # Plot the frequency
#         tag_frequency.nlargest(10).plot(kind='barh', color='skyblue', ax=ax)
        
#         ax.set_title(f'Frequency of tags in game cluster {clust}', fontsize=10)
#         ax.set_xlabel('Frequency', fontsize=8) 
#         ax.set_ylabel('Tags', fontsize=8)
#         ax.tick_params(axis='both', which='major', labelsize=7)
#         ax.set_xticks(ax.get_xticks())

#         clust += 1

# plt.tight_layout()
# plt.show()

---
User type matrix

In [ ]:
# create pivot table: rows users, columns categories, 1 if user reviewed game of this categ, else 0
user_type = user_game_type[["User id", "Type"]].drop_duplicates().pivot_table(index ="User id", columns="Type", aggfunc='size', fill_value=0)

In [ ]:
plt.matshow(user_type)
plt.title("Users and game categories")
plt.colorbar()

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
# hierarchical clustering on user_type
# check the different methods
linkage_matrix=linkage(user_type.T,method='ward', optimal_ordering=True)

plt.figure(figsize=(12,9))
dend = dendrogram(linkage_matrix, labels=user_type.head(200).columns, leaf_font_size=5, orientation="right")
plt.title("Hierarchical clustering Dendogram on user_type")
plt.ylabel('Game categories')
plt.xlabel('Distance between categories based on users')

# highlight specific labels
highlight_labels = ['Jeux de plateau', 'Jeux de cartes']

for label in plt.gca().get_ymajorticklabels():
    if label.get_text() in highlight_labels:
        label.set_color('red')
        label.set_fontweight('bold')

plt.show()

In [ ]:
# silhouette score evaluation user_type hirarchical
scores = [silhouette_score(user_type.T,fcluster(linkage(user_type.T, method='ward'),i,criterion='maxclust')) for i in range(2,10)]
scores

In [ ]:
# bi/co clustering user_type
from sklearn.cluster import SpectralBiclustering, SpectralCoclustering

fig, ax = plt.subplots(1,2, figsize=(8, 9))

# Co clustering
clustering = SpectralCoclustering(n_clusters=2, random_state=0).fit(user_type)
# group the clusters
reordered_rows = user_type.iloc[np.argsort(clustering.row_labels_)]
reordered_data = reordered_rows.iloc[:, np.argsort(clustering.column_labels_)]

cax = ax[0].matshow(reordered_data)
ax[0].set_title("Coclustering k=2 user category matrix")
fig.colorbar(cax, ax=ax[0])

# Bi clustering
clustering = SpectralBiclustering(n_clusters=2, random_state=0).fit(user_type)
# group the clusters
reordered_rows = user_type.iloc[np.argsort(clustering.row_labels_)]
reordered_data = reordered_rows.iloc[:, np.argsort(clustering.column_labels_)]

cax = ax[1].matshow(reordered_data)
ax[1].set_title("Biclustering k=2 user category matrix")
fig.colorbar(cax, ax=ax[1])

plt.subplots_adjust() 
plt.show()

In [ ]:
# silhouette score evaluation user_type, co clustering
scores_users_tags = [
    (silhouette_score(user_type, model.row_labels_), silhouette_score(user_type.T, model.column_labels_)) for i in range(2, 10) for model in [SpectralCoclustering(n_clusters=i, random_state=0).fit(user_type)]]
scores_users_tags

In [ ]:
# silhouette score evaluation user_type, bi clustering
scores_users_tags = [
    (silhouette_score(user_type, model.row_labels_), silhouette_score(user_type.T, model.column_labels_)) for i in range(2, 10) for model in [SpectralBiclustering(n_clusters=i, random_state=0).fit(user_type)]]
scores_users_tags

In [ ]:
# Clustering using dendograms user_type 
from sklearn.cluster import SpectralBiclustering, SpectralCoclustering
from scipy.cluster.hierarchy import linkage, dendrogram

# figures
fig = plt.figure(figsize=(7, 15))

ax1 = fig.add_axes([0.09, 0.1, 0.2, 0.6])
Y = linkage(user_type, method='ward') # lcustering the users 
Z1 = dendrogram(Y, orientation='left') 
ax1.set_xticks([])
ax1.set_yticks([])

ax2 = fig.add_axes([0.3, 0.71, 0.6, 0.2])
Y = linkage(user_type.T, method='ward') # clustering the categories 
Z2 = dendrogram(Y)
ax2.set_xticks([])
ax2.set_yticks([])

axmatrix = fig.add_axes([0.3, 0.1, 0.6, 0.6])
idx1 = Z1['leaves']
idx2 = Z2['leaves']
D = user_type.iloc[:,idx2]
D = D.iloc[idx1,:]
im = axmatrix.matshow(D, aspect='auto', origin='lower')

axcolor = fig.add_axes([0.91, 0.1, 0.02, 0.6])
plt.colorbar(im, cax=axcolor)
print("Hierarchical clustering on users and categories")
plt.show()

In [ ]:
# visualise clusters from linkage, hierarchical clustering 
from scipy.cluster.hierarchy import fcluster

# Categories cluster with user-categories
linkage_matrix = linkage(user_type.T, method='ward', optimal_ordering=True)
fl = fcluster(linkage_matrix,6,criterion='maxclust')
plt.title("Tag repartition in clusters")
plt.xlabel("Cluster")
plt.ylabel("Number of tags")
plt.hist(fl, alpha = 0.6)

for i in range(1, max(fl) + 1):
    print("cluster ", i)
    print(user_type.T.index[np.where(fl == i)[0]])

In [ ]:
# visualise the clusters from coclustering
clustering = SpectralCoclustering(n_clusters=7, random_state=0).fit(user_type)
for i in np.unique(clustering.column_labels_):
    print('cluster ', i)
    print(user_type.T.index[np.where(clustering.column_labels_ == i)[0]])

Tags frequence depending of users clusters also don't show differences 

---
Game type matrix

In [ ]:
game_type = user_game_type[["Game id", "Type"]].drop_duplicates().pivot_table(index ="Game id", columns="Type", aggfunc='size', fill_value=0)
game_type_old = game_type
tag_frequencies = game_type.sum(axis=0)  # Frequency of each tag across all games, to penalize popular tags
game_type = game_type.div(tag_frequencies, axis=1)

In [ ]:
plt.matshow(game_type_old)
plt.title("Games and categories")
plt.colorbar()

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

# hierarchical clustering of categories
linkage_matrix=linkage(game_type_old.T,method='ward') #optimal_ordering=True

plt.figure(figsize=(12,9))
dendrogram(linkage_matrix, labels=game_type_old.head(200).columns, leaf_font_size=5, orientation="right")
plt.title("Hierarchical clustering Dendogram on game_type")
plt.ylabel('Game categories')
plt.xlabel('Distance between categories based on games')

# highlight specific labels
highlight_labels = ['Jeux de plateau', 'Jeux de cartes']

for label in plt.gca().get_ymajorticklabels():
    if label.get_text() in highlight_labels:
        label.set_color('red')
        label.set_fontweight('bold')

plt.show()

In [ ]:
# silhouette score evaluation game_type, tags and game
scores = [(silhouette_score(game_type.T,fcluster(linkage(game_type.T, method='ward'),i,criterion='maxclust')), silhouette_score(game_type,fcluster(linkage(game_type, method='ward'),i,criterion='maxclust'))) for i in range(2,10)]
scores

In [ ]:
# hierarchical clustering, categories clusters with game-categories
# from scipy.cluster.hierarchy import fcluster
# linkage_matrix = linkage(game_type.T, method='ward', optimal_ordering=True)
# fl2 = fcluster(linkage_matrix,4,criterion='maxclust')
# plt.hist(fl2, alpha = 0.6)

# for i in range(1, max(fl2) + 1):
#     print("cluster ", i)
#     print(game_type.columns[np.where(fl2 == i)])

Tags clustering isolate not popular tags here

In [ ]:
# hierarchical clustering, categories of clustered games with game-categories

linkage_matrix = linkage(game_type, method='ward')
fl2 = fcluster(linkage_matrix,6,criterion='maxclust')
# plt.title("Repartition of game in clusters")
# plt.hist(fl2, alpha = 0.6)

fig, axes = plt.subplots(2, 3, figsize=(10, 7))
clust = 1

for i in range(3):
    for j in range(3):
        if clust > max(fl2):
            break
        
        ax = axes[i, j]  # Get the axis for each subplot
        
        # Filter the DataFrame based on the current value of fl2 (clust)
        cluster_cat = game_type.loc[:, (game_type.loc[game_type.index[np.where(fl2 == clust)]] != 0).any(axis=0)] # tags of games in the cluster
        
        # Calculate the frequency of tags in the current cluster
        # most_pop_clusters = ["Jeux de plateau", "Jeux de cartes", "Hasard (Dé, Cartes, ...)", "Placement", "Affrontement", "Combinaison"]
        # cluster_cat = cluster_cat.drop(columns = most_pop_clusters, errors='ignore')
        
        tag_frequency = (cluster_cat != 0).sum(axis=0) / (cluster_cat != 0).sum().sum()

        # Plot the frequency of 10 largest (case too much tags)
        tag_frequency.nlargest(20).plot(kind='barh', color='skyblue', ax=ax) 
        
        # Set the title and axis labels with appropriate fontsize
        ax.set_title(f'Frequency of tags in game cluster {clust}', fontsize=10)
        ax.set_xlabel('Frequency', fontsize=8) 
        ax.set_ylabel('Tags', fontsize=8)
        ax.tick_params(axis='both', which='major', labelsize=7)
        ax.set_xticks(ax.get_xticks())

        clust += 1

plt.tight_layout()
plt.show()

Hierarchical clustering on games, then plot frequence of games' tag. We can see quite distinct clusters

In [ ]:
# bi/co clustering game_type
from sklearn.cluster import SpectralBiclustering, SpectralCoclustering
fig, ax = plt.subplots(1,2, figsize=(9, 9))

# Co clustering
clustering = SpectralCoclustering(n_clusters=2, random_state=0).fit(game_type_old)
# group the clusters
reordered_rows = game_type_old.iloc[np.argsort(clustering.row_labels_)]
reordered_data = reordered_rows.iloc[:, np.argsort(clustering.column_labels_)]

cax = ax[0].matshow(reordered_data)
ax[0].set_title("Co clustering k=2 user category matrix")
fig.colorbar(cax, ax=ax[0])

# Bi clustering
clustering = SpectralBiclustering(n_clusters=2, random_state=0).fit(game_type_old)

# # group the clusters
reordered_rows = game_type_old.iloc[np.argsort(clustering.row_labels_)]
reordered_data = reordered_rows.iloc[:, np.argsort(clustering.column_labels_)]

cax = ax[1].matshow(reordered_data)
ax[1].set_title("Bi clustering k=2 game category matrix")

fig.colorbar(cax, ax=ax[1])
plt.subplots_adjust() 
plt.show()


In [ ]:
# bi/coclustering visualise tags frequency among game clusters 
Nclus = 9
clustering = SpectralCoclustering(n_clusters=Nclus, random_state=0).fit(game_type) # on the games!

fig, axes = plt.subplots(3, 3, figsize=(13, 8))
clust = 0
for i in range(3):
    for j in range(3):
        if clust >= Nclus:
            break
        
        ax = axes[i, j]  # Get the axis for each subplot
        
        # Filter the DataFrame based on the current value of fl2 (clust)
        cluster_cat = game_type.loc[:, (game_type.loc[game_type.index[np.where(clustering.row_labels_ == clust)]] != 0).any(axis=0)]

        # Calculate the frequency of tags in the current cluster
        pop_tags = ["Jeux de plateau", "Jeux de cartes", "Hasard (Dé, Cartes, ...)", "Placement", "Affrontement", "Combinaison"]
        cluster_cat = cluster_cat.drop(columns=pop_tags, errors='ignore')
        tag_frequency = (cluster_cat != 0).sum(axis=0) / (cluster_cat != 0).sum().sum()

        # Plot the frequency
        tag_frequency.nlargest(20).plot(kind='barh', color='skyblue', ax=ax)
        
        ax.set_title(f'Frequency of tags in game cluster {clust}', fontsize=10)
        ax.set_xlabel('Frequency', fontsize=8) 
        ax.set_ylabel('Tags', fontsize=8)
        ax.tick_params(axis='both', which='major', labelsize=7)
        ax.set_xticks(ax.get_xticks())

        clust += 1

plt.tight_layout()
plt.show()

In [ ]:
valeurs, nb = np.unique(clustering.row_labels_, return_counts=True)
valeurs, nb

Popular tags where dropped as they appeared as the most popular in the majority of clusters. We have some distinct clusters.

In [ ]:
# cluster 0 3 5 6

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
clust_nb = [0, 3, 5, 6]
clust = 0
for i in range(3):
    for j in range(3):
        if clust >= len(clust_nb):
            break
        
        ax = axes[i, j]  # Get the axis for each subplot
        
        # Filter the DataFrame based on the current value of fl2 (clust)
        cluster_cat = game_type.loc[:, (game_type.loc[game_type.index[np.where(clustering.row_labels_ == clust_nb[clust])]] != 0).any(axis=0)]

        # Calculate the frequency of tags in the current cluster
        pop_tags = ["Jeux de plateau", "Jeux de cartes", "Hasard (Dé, Cartes, ...)", "Placement", "Affrontement", "Combinaison", "Gestion",
                    "Gestion de main", "Majorité", "Déplacement", "Bluff", "Médiéval", "Choix simultanés", "Prise de risque", "Ambiance"]
        cluster_cat = cluster_cat.drop(columns=pop_tags, errors='ignore')
        tag_frequency = (cluster_cat != 0).sum(axis=0) / (cluster_cat != 0).sum().sum()

        # Plot the frequency
        tag_frequency.nlargest(20).plot(kind='barh', color='skyblue', ax=ax)
        
        ax.set_title(f'Frequency of tags in game cluster {clust_nb[clust]}', fontsize=10)
        ax.set_xlabel('Frequency', fontsize=8) 
        ax.set_ylabel('Tags', fontsize=8)
        ax.tick_params(axis='both', which='major', labelsize=7)
        ax.set_xticks(ax.get_xticks())

        clust += 1

plt.tight_layout()
plt.show()

# cluster 0 and 6, cluster 3 and 5 are similar

In [ ]:
# silhouette score evaluation game_type, co clustering
scores_users_tags = [
    (silhouette_score(game_type, model.row_labels_), silhouette_score(game_type.T, model.column_labels_)) for i in range(2, 10) for model in [SpectralCoclustering(n_clusters=i, random_state=0).fit(game_type)]]
scores_users_tags


In [ ]:
# silhouette score evaluation game_type, co clustering
scores_users_tags = [
    (silhouette_score(game_type, model.row_labels_), silhouette_score(game_type.T, model.column_labels_)) for i in range(2, 10) for model in [SpectralBiclustering(n_clusters=i, random_state=0).fit(game_type)]]
scores_users_tags


#### Calculate the centroid of each cluster, and find the game closest to that point

In [ ]:
# take back our 9 clusters
from sklearn.metrics import pairwise_distances

game_type["Cluster"] = clustering.row_labels_
game_type_old["Cluster"] = clustering.row_labels_
centroids = []

for clust in range(9): # find the game closest to the centroid of cluster
    # df_clust = game_type.loc[game_type.index[np.where(clustering.row_labels_ == clust)]]
    df_clust = game_type[game_type["Cluster"] == clust]
    df_clust_np = df_clust.to_numpy()

    centroid = np.mean(df_clust_np, axis = 0)
    centroids.append(centroid)
    centroid = centroid.reshape(1,-1)

    distances = pairwise_distances(df_clust_np, centroid).flatten() # eucl metric
    
    closest_game = game_type.loc[df_clust.index[np.argsort(distances)[0]]]

    # print(f"Cluster {clust} Game id {df_clust.index[np.argsort(distances)[0]]}, tags {closest_game[closest_game > 0].index} \n")


In [ ]:
# recalculate frequency based on that
# freq of tags in the cluster with centroid ponderation in game type 
clust_freq_tags = []
for i in game_type["Cluster"].unique():
    df_clust = game_type[game_type["Cluster"] == i].drop(columns=["Cluster"])
    distances_pond = 1 / (pairwise_distances(df_clust, centroids[i][:-1].reshape(1, -1)).flatten() + 1e-6)
    df_clust = df_clust.mul(distances_pond, axis = 0) # ponderate
    freq = df_clust.mean(axis=0) 
    freq /= freq.sum()
    freq["Cluster"] = i
    clust_freq_tags.append(freq)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 10))

freq_df = pd.DataFrame(clust_freq_tags, columns=clust_freq_tags[0].index.to_numpy())

c = 0
for i in range(3):
    for j in range(3):
        ax = axes[i,j]  # Get the axis for each subplot
        freq_df[freq_df["Cluster"] == c].iloc[:,:-1].sum().nlargest(10).plot(kind='barh', color='skyblue', ax=ax)
        ax.set_title(f'Frequency of tags in game cluster {freq_df[freq_df["Cluster"] == c].iloc[:,-1]}', fontsize=10)
        ax.set_xlabel('Frequency', fontsize=8) 
        ax.set_ylabel('Tags', fontsize=8)
        ax.tick_params(axis='both', which='major', labelsize=7)
        ax.set_xticks(ax.get_xticks())
        c += 1

plt.tight_layout()
plt.show()


Let's apply PCA to reduce dimension in **game type old** matrix, and project our clusters

In [ ]:
# apply PCA to see the clusters
from sklearn.decomposition import PCA

Ncompo = 3
Ncluster = 3

pca = PCA(n_components=Ncompo)
user_type_2d = pca.fit_transform(game_type_old.drop(columns=["Cluster"]))


clustering = SpectralBiclustering(n_clusters=Ncluster, random_state=0).fit(user_type_2d) # on the games
game_type_old["Cluster"] = clustering.row_labels_

centroids = []
for clust in range(Ncluster): # find the game closest to the centroid of cluster
    df_clust = game_type_old[game_type_old["Cluster"] == clust]
    df_clust_np = df_clust.to_numpy()[:,:-1]
    centroid = np.mean(df_clust_np, axis = 0)
    centroids.append(centroid)
    centroid = centroid.reshape(1,-1)

    distances = pairwise_distances(df_clust_np, centroid).flatten() # eucl metric
    closest_game = game_type_old.loc[df_clust.index[np.argsort(distances)[0]]]

print(pca.explained_variance_ratio_)


In [ ]:
centroids_pca = pca.transform(centroids)

In [ ]:
import seaborn as sns
fig, axes = plt.subplots(Ncompo, Ncompo, figsize=(15, 15))

pca_df = pd.DataFrame(user_type_2d, columns=[f'PC{i+1}' for i in range(Ncompo)])
pca_df['Cluster'] = clustering.row_labels_  # Add cluster labels to the DataFrame

# Loop through each pair of principal components 
for i in range(Ncompo):
    for j in range(i + 1):
        if i == j:
            sns.kdeplot(data=pca_df, x=f"PC{i+1}", multiple="stack", ax = axes[i, j], hue="Cluster", palette="viridis", alpha=0.6)
            ax.set_xlabel(f'PC{i+1}')
            ax.set_ylabel('Density')
        else:
            axes[i, j].scatter(user_type_2d[:, j], user_type_2d[:, i], alpha=0.7, c=clustering.row_labels_)
            axes[i, j].scatter(centroids_pca[:, j], centroids_pca[:, i], alpha=0.7, c='red')

            axes[i, j].set_xlabel(f'PC{j+1}')
            axes[i, j].set_ylabel(f'PC{i+1}')
        
        # Hide axis ticks 
        axes[i, j].tick_params(axis='both', which='both', length=0)
        
plt.suptitle('PCA Pairwise Plot', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust the layout to fit the suptitle
plt.show()

In [ ]:
sns.heatmap(centroids_pca)
plt.xlabel('PC')
plt.ylabel('Clusters centroids')
plt.show()

In [ ]:
# freq of tags in the cluster with centroid ponderation in game type old
clust_freq_tags = []
for i in game_type_old["Cluster"].unique():
    df_clust = game_type_old[game_type_old["Cluster"] == i].drop(columns=["Cluster"])
    distances_pond = 1 / pairwise_distances(df_clust, centroids[i].reshape(1, -1)).flatten()
    df_clust = df_clust.mul(distances_pond, axis = 0) # ponderate
    freq = df_clust.mean(axis=0) 
    freq /= freq.sum()
    freq["Cluster"] = i
    clust_freq_tags.append(freq)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 6))

freq_df = pd.DataFrame(clust_freq_tags, columns=clust_freq_tags[0].index.to_numpy())

for j in range(3):
    ax = axes[j]  # Get the axis for each subplot
    
    freq_df.iloc[j,:-1].nlargest(20).plot(kind='barh', color='skyblue', ax=ax)
    
    ax.set_title(f'Frequency of tags in game cluster {freq_df.iloc[j,-1]}', fontsize=10)
    ax.set_xlabel('Frequency', fontsize=8) 
    ax.set_ylabel('Tags', fontsize=8)
    ax.tick_params(axis='both', which='major', labelsize=7)
    ax.set_xticks(ax.get_xticks())

    clust += 1

plt.tight_layout()
plt.show()


---
## Clusters kmeans

### Users

In [ ]:
# apply PCA to see the clusters
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

pca = PCA(n_components=3)
user_type_2d = pca.fit_transform(user_type)

# clustering the users
# Perform K-Means clustering
kmeans = KMeans(n_clusters=3)  
clustered = kmeans.fit_predict(user_type_2d)

# plot view 1
plt.scatter(user_type_2d[:,0], user_type_2d[:,1], c=clustered)
plt.title("Cluster k=3 of users by categories, applying PCA n=3")
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

# plot view 2
# plt.scatter(user_type_2d[:,0], user_type_2d[:,2], c=clustered)
# plt.title("Cluster k=3 of users by categories, applying PCA n=2")
# plt.xlabel('PC1')
# plt.ylabel('PC3')

# plot view 3
# plt.scatter(user_type_2d[:,1], user_type_2d[:,2], c=clustered)
# plt.title("Cluster k=3 of users by categories, applying PCA n=2")
# plt.xlabel('PC2')
# plt.ylabel('PC3')
# plt.show()

We can see there aren't clusters among users. It won't be interesting to separate users on that.

### Games

In [ ]:
# apply PCA to see the clusters
pca = PCA(n_components=2)
game_type_2d = pca.fit_transform(game_type)

# clustering the games
# Perform K-Means clustering
kmeans = KMeans(n_clusters=2)  
clustered = kmeans.fit_predict(game_type_2d)

# plot view 1
plt.scatter(game_type_2d[:,0], game_type_2d[:,1], c=clustered)
plt.title("Cluster k=2 of games by categories, applying PCA n=2")
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

In [ ]:
# visualisation in 1D
pca = PCA(n_components=1)
game_type_2d = pca.fit_transform(game_type)

fig = plt.figure()
plt.scatter(game_type_2d, np.zeros(len(game_type_2d)),c=clustered, s=1)
plt.title("PCA n=1 Games clusters k=2")
plt.xlabel('PC1')

plt.show()

In [ ]:
# We inspect the tags associated to the games of both clusters to see what causes the separation 
class0 = np.where(clustered == 0)[0]
class1 = np.where(clustered == 1)[0]

# all tags associated to the games with count, choosing the most popular as the representant of the cluster
class0_label = game_type.iloc[class0].sum().sort_values(ascending=False).index[0]
class1_label =game_type.iloc[class1].sum().sort_values(ascending=False).index[0]

game_type.iloc[class0].sum().sort_values(ascending=False), game_type.iloc[class1].sum().sort_values(ascending=False)

In [ ]:
# visualisation 
pca = PCA(n_components=1)
game_type_2d = pca.fit_transform(game_type)

fig = plt.figure()
plt.hist(game_type_2d[np.where(clustered == 0)[0]], bins=100, color="red", alpha=0.5, label=class0_label)
plt.hist(game_type_2d[np.where(clustered == 1)[0]], bins=100, color="orange", alpha=0.5, label=class1_label)
plt.title("PCA n=1 Games clusters k=2")
plt.xlabel('PC1')
plt.ylabel("Number of games")
plt.legend()
plt.show()

In [ ]:
# verify the matrix is working okay
# tmp2[tmp2["User id"] == 0]['Type'].unique().shape
# np.unique([el for sub in avis_jeux[avis_jeux["User id"] == 0]["Type"].str.split(r'[|/]').tolist() for el in sub]).shape
# user_type.loc[0].sum()
# len(user_game_type[user_game_type["User id"] == 1]["Type"].unique())

## Bi clustering/Co Clustering

In [ ]:
# for reordering columns
# original_columns = user_type.columns
# original_rows = user_type.index
# new_column_order = np.argsort(clustering.column_labels_)
# new_row_order = np.argsort(clustering.row_labels_)

# Mapping new order to original columns
# reordered_column_names = original_columns[new_column_order]
# reordered_row_names = original_rows[new_row_order]
# # print("original", original_rows)
# print("new",reordered_row_names)

In [ ]:
from sklearn.mixture import GaussianMixture

# Fit a Gaussian Mixture Model to the user-item matrix (after filling NaN values)
gmm = GaussianMixture(n_components=5, random_state=0)
user_clusters = gmm.fit_predict(user_game_ratings.toarray())

# Now user_clusters contains the soft cluster assignments (users can belong to multiple clusters)
user_clusters

In [ ]:
plt.hist(user_clusters)
from collections import Counter
Counter(user_clusters)